In [1]:
#Import the required packages
#Import package pandas for data analysis
import pandas as pd


In [2]:
def build_dqr_table(df, columns):
    """
    Build and save a Data Quality Report table.
    
    Parameters:
    -----------
    df : pandas.DataFrame
    columns : Index
        Features to include
    csv_filename : str
        Output filename
    
    Returns:
    --------
    pandas.DataFrame
        Complete DQR table
    """
    stats = df[columns].describe().T
    perc_missing = 100 * (df[columns].isnull().sum() / len(df))
    cardinality = df[columns].nunique()
    
    dqr_table = pd.concat([
        stats,
        pd.DataFrame(perc_missing, columns=['%missing']),
        pd.DataFrame(cardinality, columns=['cardinality'])
    ], axis=1)
    
    return dqr_table


print("✓ Helper function loaded successfully")

✓ Helper function loaded successfully


In [3]:
def load_data(filename):
    # Reading from a csv file, into a data frame with error handling
    try:
        # If the csv file has any special symbols, it helps to specify the file encoding.
        df = pd.read_csv(filename, 
                        keep_default_na=True, 
                        delimiter=',', 
                        skipinitialspace=True)
        
        print(f"✓ Successfully loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns")
        
        # Show the first few rows in the data frame
        display(df.head(5))
        
    except FileNotFoundError:
        print("ERROR: The file 'ppr-group-25204989-train.csv' was not found.")
        print("Please ensure the file is in the current working directory.")
    except Exception as e:
        print(f"ERROR: An unexpected error occurred: {e}")

    return df

In [4]:
# Validate that the dataframe has data and expected columns
def validate_data(df):
    print("Dataset Validation Checks:")
    print("="*60)

    # Check 1: Dataset is not empty
    if df.empty:
        print("⚠ WARNING: Dataset is empty!")
    else:
        print(f"✓ Dataset contains {df.shape[0]:,} rows")

    # Check 2: Check for duplicate rows
    duplicate_count = df.duplicated().sum()
    if duplicate_count > 0:
        print(f"⚠ WARNING: Found {duplicate_count} duplicate rows")
    else:
        print("✓ No duplicate rows found")

    # Check 3: Check if all columns are unnamed
    unnamed_cols = [col for col in df.columns if 'Unnamed' in str(col)]
    if unnamed_cols:
        print(f"⚠ WARNING: Found {len(unnamed_cols)} unnamed columns: {unnamed_cols}")
    else:
        print("✓ All columns have names")

    # Check 4: Basic column overview
    print(f"\n✓ Dataset has {df.shape[1]} columns:")
    print(f"  - Column names: {list(df.columns)}")

## 1. ALF01.csv

In [5]:
df=load_data("QLF59.csv")

✓ Successfully loaded dataset with 3780 rows and 10 columns


,STATISTIC,Statistic Label,TLIST(Q1),Quarter,C04004V04762,Citizenship,C04498V05282,NACE Rev 2.1 Economic Sector,UNIT,VALUE
0,QLF59C01,Persons aged 15 years and over in Employment,20191,2019Q1,-,All Countries,-,All NACE Economic Sectors (A-V),Thousand,NaN
1,QLF59C01,Persons aged 15 years and over in Employment,20191,2019Q1,-,All Countries,A,"Agriculture, Forestry and Fishing (A)",Thousand,NaN
2,QLF59C01,Persons aged 15 years and over in Employment,20191,2019Q1,-,All Countries,F,Construction (F),Thousand,NaN
3,QLF59C01,Persons aged 15 years and over in Employment,20191,2019Q1,-,All Countries,G,Wholesale and Retail Trade (G),Thousand,NaN
4,QLF59C01,Persons aged 15 years and over in Employment,20191,2019Q1,-,All Countries,H,Transportation and Storage (H),Thousand,NaN


In [6]:
validate_data(df)

Dataset Validation Checks:
✓ Dataset contains 3,780 rows
✓ No duplicate rows found
✓ All columns have names

✓ Dataset has 10 columns:
  - Column names: ['STATISTIC', 'Statistic Label', 'TLIST(Q1)', 'Quarter', 'C04004V04762', 'Citizenship', 'C04498V05282', 'NACE Rev 2.1 Economic Sector', 'UNIT', 'VALUE']


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3780 entries, 0 to 3779
Data columns (total 10 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   STATISTIC                     3780 non-null   object 
 1   Statistic Label               3780 non-null   object 
 2   TLIST(Q1)                     3780 non-null   int64  
 3   Quarter                       3780 non-null   object 
 4   C04004V04762                  3780 non-null   object 
 5   Citizenship                   3780 non-null   object 
 6   C04498V05282                  3780 non-null   object 
 7   NACE Rev 2.1 Economic Sector  3780 non-null   object 
 8   UNIT                          3780 non-null   object 
 9   VALUE                         1785 non-null   float64
dtypes: float64(1), int64(1), object(8)
memory usage: 295.4+ KB


In [8]:


# Convert all object columns to categorical
for col in df.select_dtypes('object').columns:
    df[col] = df[col].astype('category')

df['TLIST(Q1)']=df['TLIST(Q1)'].astype('category')


# Update column groups after type corrections
numeric_columns = df.select_dtypes(['int64', 'float64']).columns
category_columns = df.select_dtypes('category').columns

print("✓ Data types corrected:")
print(f"  - Numeric features: {len(numeric_columns)}")
print(f"  - Categorical features: {len(category_columns)}")

✓ Data types corrected:
  - Numeric features: 1
  - Categorical features: 9


In [9]:
df_numeric_dqr = build_dqr_table(df, numeric_columns)
display(df_numeric_dqr)

,count,mean,std,min,25%,50%,75%,max,%missing,cardinality
VALUE,1785.0,211.353109,460.495943,5.5,23.8,84.8,169.8,2839.3,52.777778,1218


In [10]:
df_categorical_dqr = build_dqr_table(df, category_columns)
display(df_categorical_dqr)

,count,unique,top,freq,%missing,cardinality
STATISTIC,3780,1,QLF59C01,3780,0.0,1
Statistic Label,3780,1,Persons aged 15 years and over in Employment,3780,0.0,1
TLIST(Q1),3780,30,20191,126,0.0,30
Quarter,3780,30,2019Q1,126,0.0,30
C04004V04762,3780,7,-,540,0.0,7
Citizenship,3780,7,All Countries,540,0.0,7
C04498V05282,3780,18,-,210,0.0,18
NACE Rev 2.1 Economic Sector,3780,18,Accommodation and Food Service Activities (I),210,0.0,18
UNIT,3780,1,Thousand,3780,0.0,1


In [11]:
# Count each value in every categorical column, including missing values.
for col in category_columns:
    counts = (
        df[col]
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="count")
    )

    print(f"\n{col}")
    display(counts)


STATISTIC


,value,count
0,QLF59C01,3780



Statistic Label


,value,count
0,Persons aged 15 years and over in Employment,3780



TLIST(Q1)


...

,value,count
0,-,210
1,A,210
2,ZZ210228,210
3,ZZ210224,210
4,ZZ210221,210
5,ZZ210023,210
6,ZZ210011,210
7,ZZ210010,210
8,R,210
9,Q,210



NACE Rev 2.1 Economic Sector


,value,count
0,Accommodation and Food Service Activities (I),210
1,Administrative and Support Service Activities (O),210
2,Transportation and Storage (H),210
3,Services (G-V),210
4,Public Administration and Defence; Compulsory ...,210
5,"Professional, Scientific and Technical Activit...",210
6,Other NACE Activities (S to V),210
7,NACE Unknown,210
8,"Information and Communication (J,K)",210
9,Industry and Construction (B-F),210



UNIT


,value,count
0,Thousand,3780


In [12]:
for col in category_columns:
    missing_stats = (
        df.groupby(col, observed=True, dropna=False)["VALUE"]
        .agg(
            total_rows="size",
            valid_rows="count",
        )
    )

    missing_stats["missing_rows"] = (
        missing_stats["total_rows"] - missing_stats["valid_rows"]
    )
    missing_stats["missing_pct"] = (
        missing_stats["missing_rows"]
        .div(missing_stats["total_rows"])
        .mul(100)
        .round(2)
    )

    print(f"\n{col}")
    display(missing_stats.sort_values("missing_pct", ascending=False))


STATISTIC


,total_rows,valid_rows,missing_rows,missing_pct
STATISTIC,,,,
QLF59C01,3780,1785,1995,52.78



Statistic Label


,total_rows,valid_rows,missing_rows,missing_pct
Statistic Label,,,,
Persons aged 15 years and over in Employment,3780,1785,1995,52.78



TLIST(Q1)


...

,total_rows,valid_rows,missing_rows,missing_pct
C04498V05282,,,,
ZZ219990,210,33,177,84.29
A,210,44,166,79.05
P,210,55,155,73.81
Q,210,75,135,64.29
ZZ210221,210,79,131,62.38
H,210,81,129,61.43
N,210,91,119,56.67
F,210,91,119,56.67
ZZ210224,210,92,118,56.19



NACE Rev 2.1 Economic Sector


,total_rows,valid_rows,missing_rows,missing_pct
NACE Rev 2.1 Economic Sector,,,,
NACE Unknown,210,33,177,84.29
"Agriculture, Forestry and Fishing (A)",210,44,166,79.05
Public Administration and Defence; Compulsory Social Security (P),210,55,155,73.81
Education (Q),210,75,135,64.29
"Financial, Insurance and Real Estate Activities (L,M)",210,79,131,62.38
Transportation and Storage (H),210,81,129,61.43
Construction (F),210,91,119,56.67
"Professional, Scientific and Technical Activities (N)",210,91,119,56.67
Other NACE Activities (S to V),210,92,118,56.19



UNIT


,total_rows,valid_rows,missing_rows,missing_pct
UNIT,,,,
Thousand,3780,1785,1995,52.78
